# 🔬 AI Research Assistant – Multi-Agent Workflow

## Scenario

This notebook implements a **Multi-Agent Research Assistant** that automates the research-to-report pipeline:

| Agent | Role | Tools Used |
|-------|------|------------|
| **Researcher Agent** | Searches the web and Wikipedia to gather comprehensive, up-to-date information on a given topic | DuckDuckGo Search, Wikipedia Search |
| **Writer Agent** | Synthesises the raw findings into a polished, structured research report | *(LLM only)* |

## Workflow

```
START ──► researcher_node ──► writer_node ──► [INTERRUPT] ──► human_review_node ──► END
                                                                       │
                                                              (if revision needed)
                                                                       │
                                                               writer_node ◄──┘
```

**Human-in-the-Loop (HITL):** Before the final report is approved, the graph pauses and asks the human reviewer to either approve the draft or request revisions. The workflow loops until the reviewer is satisfied (max 3 revision rounds).

**Memory:** A `MemorySaver` checkpointer persists the full conversation state across HITL interrupts, enabling seamless resumption.

---
**Assignment:** AI Agents and Workflows for Developers – April 2026 Individual Project  
**Model:** OpenAI GPT-4o-mini  
**Framework:** LangGraph + LangChain

In [ ]:
# Install required packages
!pip install -q \
    "langchain>=0.3.0" \
    "langchain-openai>=0.2.0" \
    "langchain-community>=0.3.0" \
    "langgraph>=0.2.0" \
    "duckduckgo-search>=6.0.0" \
    wikipedia
print("All packages installed successfully.")

## Imports & Configuration

In [ ]:
import os
import uuid
from typing import TypedDict, Annotated, List, Literal

from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage,
    SystemMessage, ToolMessage
)
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

print("Imports loaded successfully.")

In [ ]:
# ── API Key Setup ────────────────────────────────────────────────────────────
# In Google Colab: add OPENAI_API_KEY to Secrets (key icon in left sidebar)
# Locally: set it as an environment variable before running
# DO NOT paste your key directly into this cell!

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        raise EnvironmentError(
            "OPENAI_API_KEY not found. "
            "Set it as a Colab Secret or environment variable."
        )
    print("API key loaded from environment variable.")

# DuckDuckGo and Wikipedia require no API keys
print("DuckDuckGo & Wikipedia: no API key required.")

## State Definition

In [ ]:
class ResearchState(TypedDict):
    """Shared state passed between all agents in the graph."""
    user_request:   str                                    # Original topic from the user
    search_results: str                                    # Raw findings from Researcher
    draft_report:   str                                    # Current draft from Writer
    final_report:   str                                    # Approved final report
    human_feedback: str                                    # Reviewer's feedback
    revision_count: int                                    # Number of revision cycles
    messages: Annotated[List[BaseMessage], add_messages]   # Accumulated conversation log

print("ResearchState defined.")

## Tools Setup

In [ ]:
# Tool 1 – DuckDuckGo web search (no API key required)
ddg_search = DuckDuckGoSearchRun(
    name="web_search",
    description="Search the web with DuckDuckGo for current news and facts."
)

# Tool 2 – Wikipedia (no API key required)
wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=3000)
wikipedia_tool = WikipediaQueryRun(
    api_wrapper=wiki_api,
    name="wikipedia_search",
    description="Search Wikipedia for comprehensive background knowledge."
)

TOOLS = [ddg_search, wikipedia_tool]
TOOLS_DICT = {t.name: t for t in TOOLS}

print(f"Tools ready: {[t.name for t in TOOLS]}")

## LLM Instances

In [ ]:
# Researcher uses tool-calling; Writer only needs text generation
researcher_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
researcher_with_tools = researcher_llm.bind_tools(TOOLS)

writer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

print("LLM instances ready.")

## Agent Nodes

In [ ]:
# ─────────────────────────────────────────────────────────────
# AGENT 1 – Researcher
# System prompt: expert researcher who uses DuckDuckGo + Wikipedia
# ─────────────────────────────────────────────────────────────
def researcher_node(state: ResearchState) -> dict:
    print("\n" + "=" * 55)
    print("[Researcher Agent] Gathering information...")
    print("=" * 55)

    system_prompt = (
        "You are an expert research agent.\n"
        "Use 'web_search' for current facts and recent developments.\n"
        "Use 'wikipedia_search' for background knowledge and definitions.\n"
        "Make at least 2 searches (one of each tool) for thorough coverage.\n"
        "After searching, summarise ALL findings in a detailed, structured way."
    )

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Research this topic thoroughly: {state['user_request']}")
    ]

    for _ in range(8):  # max tool-call iterations
        response = researcher_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print("  [Researcher] Research complete.")
            break

        for tc in response.tool_calls:
            tool_name = tc["name"]
            query = str(list(tc["args"].values())[0])
            print(f"  -> {tool_name}: '{query[:70]}'")
            try:
                result = TOOLS_DICT[tool_name].invoke(tc["args"])
                result_str = str(result)
            except Exception as e:
                result_str = f"Tool error: {e}"
            messages.append(ToolMessage(content=result_str, tool_call_id=tc["id"]))

    final_ai = next(
        (m for m in reversed(messages) if isinstance(m, AIMessage) and not m.tool_calls),
        None
    )
    summary = final_ai.content if final_ai else "No research summary available."

    return {
        "search_results": summary,
        "messages": [AIMessage(content=f"Research done: {state['user_request'][:60]}")]
    }


# ─────────────────────────────────────────────────────────────
# AGENT 2 – Writer
# System prompt: expert technical writer who creates/revises reports
# ─────────────────────────────────────────────────────────────
def writer_node(state: ResearchState) -> dict:
    revision_count = state.get("revision_count", 0)
    is_revision = revision_count > 0

    print("\n" + "=" * 55)
    label = f"Revising report (revision #{revision_count})" if is_revision else "Creating initial report"
    print(f"[Writer Agent] {label}...")
    print("=" * 55)

    if is_revision:
        feedback = state.get("human_feedback", "")
        prompt = (
            f"You are an expert technical writer. Revise the report based on the reviewer's feedback.\n\n"
            f"ORIGINAL REPORT:\n{state.get('draft_report', '')}\n\n"
            f"REVIEWER FEEDBACK:\n{feedback}\n\n"
            f"RESEARCH DATA:\n{state.get('search_results', '')}\n\n"
            f"Address ALL feedback points. Keep the professional structure."
        )
    else:
        prompt = (
            f"You are an expert technical writer. Create a comprehensive research report.\n\n"
            f"TOPIC: {state['user_request']}\n\n"
            f"RESEARCH FINDINGS:\n{state.get('search_results', '')}\n\n"
            f"Format the report with these sections:\n"
            f"# Research Report: [Title]\n"
            f"## Executive Summary\n"
            f"## Key Findings (3-5 bullet points)\n"
            f"## Detailed Analysis (3-4 paragraphs)\n"
            f"## Conclusions & Implications\n"
        )

    response = writer_llm.invoke([HumanMessage(content=prompt)])
    print("  [Writer] Draft ready.")

    return {
        "draft_report": response.content,
        "messages": [AIMessage(content="Report draft created/updated.")]
    }


# ─────────────────────────────────────────────────────────────
# Human Review Node  (runs AFTER the HITL interrupt is resolved)
# ─────────────────────────────────────────────────────────────
def human_review_node(state: ResearchState) -> dict:
    feedback = state.get("human_feedback", "").strip().lower()
    revision_count = state.get("revision_count", 0)

    APPROVED = {"approve", "approved", "yes", "ok", "looks good",
                "good", "accept", "accepted", "fine", "great", "perfect", "lgtm"}

    is_approved = (
        feedback in APPROVED
        or any(kw in feedback for kw in ["approve", "looks good", "accept"])
        or revision_count >= 3
    )

    if is_approved:
        print("\n  [Human Review] Report APPROVED. Finalising...")
        return {"final_report": state["draft_report"]}
    else:
        new_count = revision_count + 1
        print(f"\n  [Human Review] Revision #{new_count} requested.")
        return {
            "revision_count": new_count,
            "human_feedback": state.get("human_feedback", "")
        }


def route_after_review(state: ResearchState) -> str:
    """Conditional router: approved -> END, revision -> writer_node."""
    return END if state.get("final_report") else "writer_node"


print("Agent nodes defined.")

## Graph Construction

In [ ]:
# Persistent checkpointer (MemorySaver keeps state across HITL interrupts)
memory_checkpointer = MemorySaver()

builder = StateGraph(ResearchState)

# Register nodes
builder.add_node("researcher_node", researcher_node)
builder.add_node("writer_node",     writer_node)
builder.add_node("human_review_node", human_review_node)

# Edges
builder.add_edge(START, "researcher_node")
builder.add_edge("researcher_node", "writer_node")
builder.add_edge("writer_node", "human_review_node")
builder.add_conditional_edges("human_review_node", route_after_review)

# Compile with MemorySaver + HITL interrupt before human_review_node
graph = builder.compile(
    checkpointer=memory_checkpointer,
    interrupt_before=["human_review_node"]
)

print("Graph compiled successfully.")
print("Flow: START -> researcher -> writer -> [INTERRUPT] -> human_review -> END")
print("      (or, if revision requested)  -> writer -> [INTERRUPT] -> ...")

## Core Function: `execute_workflow`

In [ ]:
def execute_workflow(user_request: str) -> str:
    """
    Run the multi-agent research workflow with interactive Human-in-the-Loop review.

    The graph pauses before human_review_node on each iteration. The reviewer
    types 'approve' (or similar) to finalise, or provides revision instructions
    to trigger another writer pass.

    Args:
        user_request: The research topic or question.

    Returns:
        The final approved research report as a string.
    """
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    initial_state = {
        "user_request":   user_request,
        "search_results": "",
        "draft_report":   "",
        "final_report":   "",
        "human_feedback": "",
        "revision_count": 0,
        "messages":       []
    }

    print("\n" + "#" * 60)
    print(f"  RESEARCH WORKFLOW  |  {thread_id[:8]}")
    print(f"  Topic: {user_request}")
    print("#" * 60)

    # ── Phase 1: run agents until HITL interrupt ────────────────
    print("\n[Phase 1] Launching Researcher and Writer agents...")
    for _ in graph.stream(initial_state, config, stream_mode="values"):
        pass

    # ── HITL loop ───────────────────────────────────────────────
    review_round = 0
    while True:
        snapshot = graph.get_state(config)
        if not snapshot.next:       # graph finished
            break

        review_round += 1
        draft = snapshot.values.get("draft_report", "(no draft)")

        print("\n" + "-" * 60)
        print(f"  HUMAN-IN-THE-LOOP REVIEW  (round {review_round})")
        print("-" * 60)
        print(draft)
        print("-" * 60)
        print("\nType 'approve' to finalise, or enter revision instructions:")

        human_input = input("Your feedback: ").strip() or "approve"

        graph.update_state(config, {"human_feedback": human_input})

        print(f"\n[Phase {review_round + 1}] Resuming graph...")
        for _ in graph.stream(None, config, stream_mode="values"):
            pass

    final = graph.get_state(config).values.get(
        "final_report", "Workflow ended without producing a final report."
    )

    print("\n" + "#" * 60)
    print("  WORKFLOW COMPLETE – FINAL REPORT")
    print("#" * 60)
    print(final)
    return final


print("execute_workflow() defined.")

## Test Helper (simulates HITL without blocking `input()`)

In [ ]:
def run_workflow_test(
    user_request: str,
    feedback_sequence: list,
    test_name: str = ""
) -> str:
    """
    Automated variant of execute_workflow for test cases.
    Instead of prompting the user, it pops values from feedback_sequence
    at each HITL checkpoint (defaults to 'approve' when the list is exhausted).

    Args:
        user_request:      Research topic.
        feedback_sequence: Ordered list of feedback strings to inject.
        test_name:         Optional label for display.

    Returns:
        The final approved report string.
    """
    if test_name:
        print("\n" + "*" * 60)
        print(f"  {test_name}")
        print("*" * 60)

    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    initial_state = {
        "user_request":   user_request,
        "search_results": "",
        "draft_report":   "",
        "final_report":   "",
        "human_feedback": "",
        "revision_count": 0,
        "messages":       []
    }

    print(f"\nTopic: {user_request}")
    print(f"Planned feedback sequence: {feedback_sequence}")

    # Run agents until HITL interrupt
    for _ in graph.stream(initial_state, config, stream_mode="values"):
        pass

    fb_idx = 0
    round_num = 0

    while True:
        snapshot = graph.get_state(config)
        if not snapshot.next:
            break

        round_num += 1
        draft = snapshot.values.get("draft_report", "")

        feedback = feedback_sequence[fb_idx] if fb_idx < len(feedback_sequence) else "approve"
        fb_idx += 1

        print(f"\n--- HITL Round {round_num} ---")
        print(f"Draft preview: {draft[:300]}..." if len(draft) > 300 else f"Draft: {draft}")
        print(f"Simulated feedback: '{feedback}'")

        graph.update_state(config, {"human_feedback": feedback})

        for _ in graph.stream(None, config, stream_mode="values"):
            pass

    final = graph.get_state(config).values.get("final_report", "No final report.")

    print(f"\n[TEST RESULT] Final report length: {len(final)} chars")
    print("\n" + "=" * 60)
    print("FINAL REPORT")
    print("=" * 60)
    print(final)
    print("=" * 60)
    return final


print("run_workflow_test() defined.")

---
# Test Cases

Five test cases demonstrating different HITL scenarios:

| # | Topic | HITL Scenario |
|---|-------|---------------|
| TC-01 | History of the World Wide Web | Direct approval |
| TC-02 | Benefits of renewable energy | One revision, then approval |
| TC-03 | How quantum computing works | Direct approval (alt keyword) |
| TC-04 | AI impact on employment | Two revisions, then approval |
| TC-05 | Microbiome & human health | Direct approval (different domain) |

In [ ]:
# ── TC-01: Direct Approval ────────────────────────────────────────────────────
# Demonstrates the happy-path: the first draft is accepted without any revisions.

result_tc01 = run_workflow_test(
    user_request="The history and evolution of the World Wide Web",
    feedback_sequence=["approve"],
    test_name="TC-01: Direct Approval"
)

In [ ]:
# ── TC-02: One Revision Then Approval ────────────────────────────────────────
# Demonstrates the reviewer requesting a change before approving.
# The writer re-runs with the feedback, produces a revised draft, which is
# then approved in round 2.

result_tc02 = run_workflow_test(
    user_request="Benefits and challenges of renewable energy sources",
    feedback_sequence=[
        "Please add specific statistics on solar and wind capacity growth, "
        "and include a brief comparison of costs versus fossil fuels.",
        "approve"
    ],
    test_name="TC-02: One Revision Then Approval"
)

In [ ]:
# ── TC-03: Approval With Alternative Keyword ─────────────────────────────────
# Shows that the system recognises synonyms for approval
# ('looks good' is treated the same as 'approve').

result_tc03 = run_workflow_test(
    user_request="How does quantum computing work and what are its potential applications?",
    feedback_sequence=["looks good"],
    test_name="TC-03: Approval With Alternative Keyword ('looks good')"
)

In [ ]:
# ── TC-04: Multiple Revisions ─────────────────────────────────────────────────
# Demonstrates two successive revision cycles before final approval.
# Round 1: reviewer asks for a healthcare-specific section.
# Round 2: reviewer asks to deepen that section.
# Round 3: reviewer approves.

result_tc04 = run_workflow_test(
    user_request="Impact of artificial intelligence on employment and the job market",
    feedback_sequence=[
        "The executive summary is too long. Please shorten it to 2 sentences "
        "and add a dedicated section on AI's impact in the healthcare job market.",
        "The healthcare section needs more depth — please add two concrete examples "
        "of AI diagnostic tools currently used in hospitals.",
        "approve"
    ],
    test_name="TC-04: Multiple Revisions (2 rounds) Then Approval"
)

In [ ]:
# ── TC-05: Different Domain (Biology / Medicine) ──────────────────────────────
# Verifies the workflow generalises beyond technology topics.

result_tc05 = run_workflow_test(
    user_request="The human microbiome: its role in health and disease",
    feedback_sequence=["accepted"],
    test_name="TC-05: Different Domain (Biology/Medicine)"
)

print("\nAll 5 test cases completed successfully.")